# R-tag pipeline — single run walkthrough

This notebook runs the **Python R-tag (RTEG) workflow** for the KB331 sample filter.

**Manual reference:** Jing Yang's SKILL flow in [`../rdsBawTEGAutoFromTemp.il`](../rdsBawTEGAutoFromTemp.il)  
**Documentation:** [`../README.md`](../README.md) · [`../CLAUDE.md`](../CLAUDE.md)

Run all cells top-to-bottom from the `python_code/` directory.

| Step | Section | Module(s) |
|---|---|---|
| 1 | Process inputs | `layermap.py`, `inspect_refs.py` |
| 2 | Selection | `separate.py` |
| 3 | Separation | `prep_resonator_ppd.py` |
| 4 | Setting up | `prep_rteg_frame.py`, `export_gds.py` |
| 5 | Routing | *(planned)* |
| 6 | Verification | *(planned)* |


In [ ]:
from __future__ import annotations
import io
import sys
from contextlib import redirect_stdout
from pathlib import Path
import gdstk
import pandas as pd


def resolve_python_code_root() -> Path:
    """Find python_code/ by looking for input_files/ + src/."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "input_files").is_dir() and (candidate / "src").is_dir():
            return candidate
    return here


ROOT = resolve_python_code_root()
SRC = ROOT / "src"
INPUT_FILES = ROOT / "input_files"
DRAFT = ROOT / "draft_output"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DRAFT.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {ROOT}")
print(f"Input files:       {INPUT_FILES}")
print(f"Draft output:      {DRAFT}")
print(f"Source code:       {SRC}")

## Define Inputs Here For Running This Notebook

In [ ]:
FILTER = INPUT_FILES / "KB331_N_01.gds" # input filter GDS file
FRAME = INPUT_FILES / "KB331_N_Frame.gds" # die frame
PPD = INPUT_FILES / "GSG_frame.gds" # GSG ppd frame
LAYERMAP = INPUT_FILES / "layermap" # Skyworks layer map

## 1. Process Inputs

In [ ]:
# Ensure all referenced input files exist, report and handle missing files

input_files = {
    "Filter layout": FILTER,
    "Frame template": FRAME,
    "Probe device": PPD,
    "Layer table": LAYERMAP,
}

input_roles = {
    "Filter layout": "Clean hierarchical filter GDS — resonators + connect metal",
    "Frame template": None,  # will fill after loading FRAME sizing
    "Probe device": "ppd_1port — GSG pad reference",
    "Layer table": "Skyworks name → GDS (layer, datatype)",
}

file_errors = []

# Check all files, prepare info for table rows
rows = []
frame_size_str = "unknown size"

# Check supporting files first so sizing is available for FRAME
for label, path in input_files.items():
    exists = path.exists()
    if not exists:
        file_errors.append(f"ERROR: {label} file not found: {path}")
    # Calculate size
    size = f"{path.stat().st_size:,} B" if exists and path.is_file() else "—"
    # Frame sizing to be updated after loading below
    rows.append({"file": label, "path": path.name, "exists": exists, "size": size, "role": input_roles[label]})

# Specifically for FRAME, if it is missing, output an error and skip sizing attempts
if FRAME.exists():
    try:
        frame_lib = gdstk.read_gds(FRAME)
        frame_cell = frame_lib.top_level()[0]
        frame_bb = frame_cell.bounding_box()
        if frame_bb is not None:
            (fx0, fy0), (fx1, fy1) = frame_bb
            frame_width = fx1 - fx0
            frame_height = fy1 - fy0
            frame_size_str = f"{frame_width:.1f}×{frame_height:.1f} µm"
    except Exception as e:
        file_errors.append(f"ERROR: Unable to read Frame template or extract bounding box: {e}")

# Update the row for FRAME with computed size string
for row in rows:
    if row["file"] == "Frame template":
        row["role"] = f"{frame_size_str} GSG probe frame (six BAW_MB2 pads)"

# If there were errors, print them, otherwise continue
if file_errors:
    for err in file_errors:
        print(err)
    print("❌ Aborting due to missing required input files ❌")
else:
    display(pd.DataFrame(rows))

## 2. Selection

2.1 start with layermap definitions

2.2 inspect layer references

2.3 start identifying resonators from filter GDS file

### 2.1 `layermap.py` — layer name lookups

Loads the Skyworks layermap file from `LAYERMAP` (defined above). Maps names like `BAW_MBE` to GDS `(layer, datatype)` pairs.

In [ ]:
from src.layermap import load_layermap

layermap = load_layermap(LAYERMAP)
print(f"Loaded {len(layermap)} layers from {LAYERMAP.name}")

for name in ("BAW_MBE", "BAW_MTE", "BAW_MB2", "BAW_EDGE"):
    layer, dt = layermap.pair(name)
    print(f"  {name:12s} -> GDS ({layer}, {dt})")

### 2.2 `inspect_refs.py` — hierarchy sanity check

Lists placed references in the filter GDS: resonators, vias, connect cells. Useful when instance names did not survive export.

In [ ]:
from src.inspect_refs import inspect_gds

buf = io.StringIO()
with redirect_stdout(buf):
    inspect_gds(FILTER)

# 
print(buf.getvalue()[:2000])
if len(buf.getvalue()) > 2000:
    print("... (truncated)")

### 2.3 `separate.py` — resonator and vtb identification

Finds placed resonators (masters: `series`, `shunt`, `rcap`, `mimcap`) and `vtb` vias under the filter parent cell. Returns dataframe-ready rows via `identify(FILTER)`.

In [ ]:
from src.separate import identify

identification = identify(FILTER)

parent = identification.parent
filter_lib = identification.library
res_list = identification.resonators
vias = identification.vias

res_df = pd.DataFrame(identification.resonator_rows()) #DF of resonators 
via_df = pd.DataFrame(identification.via_rows()) #DF of VIAs

print(f"Parent cell: {parent}")
print(f"Resonators: {len(res_list)}  |  Vias: {len(vias)}")

res_df

In [ ]:
via_df

## 3. Separation
For each resonator found place it together with the GSG ppd frame in the center

### `prep_resonator_ppd.py` — PPD + resonator

For each row in `res_df`, combine the resonator with the GSG PPD frame (`GSG_frame.gds`):

1. PPD placed at `(0, 0)`
2. Resonator bbox center aligned to the PPD bbox center
3. If resonator MBE/MTE overlaps GSG pad metal, nudge until **≥10 µm** metal clearance
4. Same nudge must also keep **BAW_ReF / BAW_CAV** release holes **≥6 µm** from the GSG frame (no touching)

Returns `ppd_assemblies` (in-memory, editable) and `ppd_files_df` (includes `min_release_clearance_um`). Step 4 (`prep_rteg_frame.py`) places these in the die frame.

In [ ]:
from IPython.display import HTML, display
from src.prep_resonator_ppd import (
    MIN_RELEASE_HOLE_CLEARANCE_UM,
    assemblies_summary_df,
    prep_resonator_ppd,
)

ppd_assemblies = prep_resonator_ppd(res_df, res_list, PPD)
ppd_files_df = assemblies_summary_df(ppd_assemblies)

print(f"Built {len(ppd_assemblies)} PPD assemblies")
display(
    ppd_files_df[
        [
            "index",
            "inst_name",
            "type",
            "clearance_shift_x",
            "clearance_shift_y",
            "min_release_clearance_um",
            "shift_x",
            "shift_y",
        ]
    ]
)

violations = ppd_files_df[
    ppd_files_df["min_release_clearance_um"] < MIN_RELEASE_HOLE_CLEARANCE_UM
]
if not violations.empty:
    display(violations)
    raise RuntimeError(
        f"Release holes closer than {MIN_RELEASE_HOLE_CLEARANCE_UM} µm to GSG frame"
    )
print(
    f"All resonators satisfy release-hole clearance "
    f"(>= {MIN_RELEASE_HOLE_CLEARANCE_UM} µm to GSG frame)"
)

simply preview of the resonator + ppd GSG placement within the notebook directly 

In [ ]:
# # Display/Preview of frame within this notebook

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="ppd-preview-item">'
#     f'<div class="ppd-preview-label">[{asm.index}] {asm.inst_name} ({asm.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in ppd_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .ppd-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .ppd-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .ppd-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .ppd-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="ppd-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

## 4. Setting Up

place the ppd+resonator combo now into the original die frame.

for the width (x axis) leave 4% margin in both sides

for the height (y axis) leave 7% margin in both sides

### `prep_rteg_frame.py` — die frame placement

For each `ppd_assembly`, place the step-2 `top_cell` inside `FRAME`:

1. Die frame at `(0, 0)`; margins measured from the **inner** MBE die frame (not the outer 460×580 bbox)
2. Assembly left edge at 3.5% inside the inner frame; Y centered with 7% margins
3. MBE filler rectangle on the right: inner-frame center → margined right edge, same height as the placed GSG/resonator assembly

Returns `rteg_assemblies` and `rteg_files_df`. Step 5 (later) adds interconnect routing and DRC.

In [ ]:
from src.prep_rteg_frame import (
    assemblies_summary_df,
    prep_rteg_in_frame,
    preview_assembly_svg,
)

rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)
rteg_files_df = assemblies_summary_df(rteg_assemblies)

print(f"Built {len(rteg_assemblies)} RTEG frame assemblies")
display(rteg_files_df)

In [ ]:
# from IPython.display import HTML, display

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="rteg-preview-item">'
#     f'<div class="rteg-preview-label">[{asm.index}] {asm.inst_name} ({asm.ppd_assembly.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in rteg_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .rteg-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .rteg-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .rteg-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .rteg-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="rteg-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

In [ ]:
from src.export_gds import export_gds, export_summary_df

rteg_export_df = export_summary_df(
    export_gds(
        rteg_assemblies,
        DRAFT,
        layermap=layermap,
        parent=parent,
        stage="framed",
    )
)

print(f"Exported {len(rteg_export_df)} GDS files to {DRAFT}")
print("Layer names: open each .gds with its matching .lyp in KLayout")
display(rteg_export_df)


========================================================

## 5. Routing - solving interconnect algorithm

**Goal:** turn a framed resonator into a DRC-clean RTEG — one fused MBE ground body with two pockets carved out (resonator + signal)

steps to solve this 

1. **Collect (5.1)** — pull ground plates, preserved MBE/MTE, release holes, frame boundary by layermap (`rteg_collect.py`).
2. **Classify (5.2)** — assign each GSG node to signal/ground from resonator type (`rteg_classify.py`).
3. **Build signal (MTE) (5.3)** — shaped connector plate from preserved MTE to the signal pad; union into one routed MTE path (`rteg_signal.py`).
4. **Union ground (5.4)** — OR the ground node blocks + filler + preserved MBE into one body (bridge gaps if needed).
5. **Carve pockets (5.5)** — subtract signal net (+14µm), resonator keep-out, and release holes (+6µm) from the body.
6. **Reconnect (5.6)** — union preserved ground metal back into the carved body; drop slivers.

### 5.1 — What the output groups mean (~30 sec)

Step 5.1 splits the framed layout into **typed polygon sets** (all layers from the layermap). Later steps only run booleans on these groups.

**Ground plates** (frame-side MBE — not resonator, not preserved filter metal)

| Group | Meaning |
|-------|---------|
| `top_ground` | Upper outer GSG pad body + inward arm (MBE). |
| `bottom_ground` | Lower outer GSG pad body + inward arm (MBE). |
| `center_node` | MBE at the middle GSG position. Signal vs ground is decided in step 5.2; here it is just “metal at the center node.” |
| `filler_plate` | Right-hand MBE rectangle from step 4 — the wide plate that fills the right half of the inner frame. |

**Preserved metal** (filter interconnect copied from the parent filter)

| Group | Meaning |
|-------|---------|
| `preserved_mbe` | Ground (MBE) from `connectMBE`, overlapping this resonator in RTEG coordinates. Ground routing must land on this (NPI). |
| `preserved_mte` | Signal (MTE) from `connectMTE`. Used to build the signal plate (5.3); ground carving must clear it (+14 µm). |

**Release holes** (process keepouts — near this resonator only)

| Group | Meaning |
|-------|---------|
| `BAW_ReF` | Release-hole outlines on ReF. |
| `BAW_CAV` | Cavity/release outlines on CAV. Subtracted from the ground body with 6 µm clearance in step 5.5. |

**Frame boundary** (reference geometry, not merged ground metal)

| Group | Meaning |
|-------|---------|
| `inner_cavity` | Inner die cavity rectangle — routable envelope inside the frame ring. |
| `frame_ring` | Die-frame MBE ring from the frame GDS. |

**Pipeline sketch:** ground plates → union → carve away signal + holes → reconnect `preserved_mbe`. Signal path: `preserved_mte` → shaped connector to the signal node.

In [ ]:
from src.rteg_collect import (
    RtegCollectConfig,
    collect_geometry_roles,
    geometry_roles_summary_table,
)

COLLECT_CONFIG = RtegCollectConfig()

all_roles: dict[int, object] = {}
collect_rows: list[dict[str, object]] = []
collect_detail_rows: list[dict[str, object]] = []

for idx in range(len(identification.resonators)):
    res = identification.resonators[idx]
    roles = collect_geometry_roles(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        config=COLLECT_CONFIG,
    )
    all_roles[idx] = roles
    counts = roles.group_counts()
    collect_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            **{k: counts[k] for k in sorted(counts)},
            "total_polygons": sum(counts.values()),
        }
    )
    collect_detail_rows.extend(geometry_roles_summary_table(roles))

collect_overview_df = pd.DataFrame(collect_rows).sort_values("index")
collect_detail_df = pd.DataFrame(collect_detail_rows)

print(f"Collected geometry for {len(all_roles)} resonators\n")
display(collect_overview_df)
print("\nPolygon detail (all resonators):")
display(collect_detail_df)

### 5.2 — Classify GSG nodes by Type

`rteg_classify.classify_nodes` labels each GSG band **signal** or **ground** from the resonator **type** (step 2).

| Resonator type | Center pad | Top pad | Bottom pad | `signal_band` |
|----------------|------------|---------|------------|---------------|
| **shunt** | signal | ground | ground | `center` |
| **series** | ground | ground | ground | `on_resonator` |

For **series**, signal does not escape to a GSG probe pad — step 5.3 builds a thin MTE strip along the resonator MBE perimeter between release holes.

`filler_plate` is always ground (not a probe node).

| Output field | Meaning |
|--------------|---------|
| `signal_band` | `center` (shunt pad) or `on_resonator` (series body) |
| `nodes[].net` | `signal` or `ground` per band |
| `method` | `res_type` |
| `note` | Human-readable rule applied |

In [ ]:
from src.rteg_classify import ClassifyNodesConfig, classify_nodes, classification_summary_table

CLASSIFY_CONFIG = ClassifyNodesConfig()

all_classify: dict[int, object] = {}
classify_overview_rows: list[dict[str, object]] = []
classify_detail_rows: list[dict[str, object]] = []

for idx, roles in all_roles.items():
    res = identification.resonators[idx]
    classification = classify_nodes(
        roles.ground_plates,
        roles.preserved,
        res_type=res.res_type,
        config=CLASSIFY_CONFIG,
    )
    all_classify[idx] = classification
    by_band = classification.by_band()
    classify_overview_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            "method": classification.method,
            "signal_band": classification.signal_band,
            "top": by_band["top"].net,
            "center": by_band["center"].net,
            "bottom": by_band["bottom"].net,
            "note": classification.note,
        }
    )
    classify_detail_rows.extend(
        classification_summary_table(
            classification,
            index=idx,
            inst_name=roles.inst_name,
            res_type=res.res_type,
        )
    )

classify_overview_df = pd.DataFrame(classify_overview_rows).sort_values("index")
classify_df = pd.DataFrame(classify_detail_rows)

print(f"Classified {len(all_classify)} resonators\n")
display(classify_overview_df)
print("\nPer-band detail (all resonators):")
display(classify_df)

# Type rule checks — all resonators
for idx, classification in all_classify.items():
    res_type = identification.resonators[idx].res_type
    by_band = classification.by_band()
    if res_type == "shunt":
        assert classification.signal_band == "center"
        assert by_band["center"].net == "signal"
        assert by_band["top"].net == "ground"
        assert by_band["bottom"].net == "ground"
    elif res_type == "series":
        assert classification.signal_band == "on_resonator"
        assert by_band["center"].net == "ground"
        assert by_band["top"].net == "ground"
        assert by_band["bottom"].net == "ground"
    else:
        raise AssertionError(f"[{idx}] unsupported res_type {res_type!r}")

print(f"\nType-rule checks passed for all {len(all_classify)} resonators")

### 5.3 — Build signal (MTE) net (~30 sec)

**Shunt:** `build_signal_net` connects preserved filter **MTE** to the center **signal** GSG pad with a shaped connector plate, then boolean-unions preserved MTE + connector.

**Series:** a thin MTE strip along the resonator **MBE perimeter** between two release holes (`on_resonator`) — no pad-directed connector, no pad reach.

| Function | Role |
|----------|------|
| `signal_endpoints` | Shunt: launch points between preserved MTE and signal pad |
| `build_signal_plate` | Shunt: 45/90° connector stroke (straight / L / 45 / Z) |
| `build_series_boundary_mte` | Series: stroked perimeter arc between release holes |
| `union_signal_net` | Shunt: OR preserved MTE + connector |
| `build_signal_net` | Orchestrator — branches on `signal_band` |

**Done when (shunt):** connector merges with preserved MTE and reaches the signal pad; connector DRC-clean vs ground MBE.

**Done when (series):** non-empty perimeter strip; perimeter arc centerline (`shape_name = on_resonator`); strip area ≪ resonator MBE body.


In [ ]:
from src.rteg_signal import (
    SignalBuildConfig,
    build_signal_net,
    signal_net_summary_table,
)

SIGNAL_CONFIG = SignalBuildConfig()

all_signal: dict[int, object] = {}
signal_overview_rows: list[dict[str, object]] = []
signal_detail_rows: list[dict[str, object]] = []

for idx, roles in all_roles.items():
    res = identification.resonators[idx]
    classification = all_classify[idx]
    signal = build_signal_net(
        roles.preserved,
        classification,
        roles.ground_plates,
        layermap,
        config=SIGNAL_CONFIG,
        res=res,
        assembly=rteg_assemblies[idx],
        release_holes=roles.release_holes,
    )
    all_signal[idx] = signal
    summary = signal.summary()
    signal_overview_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            "signal_band": classification.signal_band,
            "shape": summary["shape"],
            "n_net_polygons": summary["n_net_polygons"],
            "is_connected": summary["is_connected"],
            "reaches_pad": summary["reaches_pad"],
            "is_success": summary["is_success"],
            "min_ground_spacing_um": summary["min_ground_spacing_um"],
            "drc_violations": summary["drc_violations"],
        }
    )
    for row in signal_net_summary_table(signal):
        row["index"] = idx
        row["inst_name"] = roles.inst_name
        row["res_type"] = res.res_type
        signal_detail_rows.append(row)

signal_overview_df = pd.DataFrame(signal_overview_rows).sort_values("index")
signal_detail_df = pd.DataFrame(signal_detail_rows)

print(f"Built signal (MTE) nets for {len(all_signal)} resonators\n")
display(signal_overview_df)
print("\nSignal net detail (all resonators):")
display(signal_detail_df)

PAD_ROUTE_SHAPES = {"straight", "route_L", "route_45", "route_Z"}

not_connected = signal_overview_df[~signal_overview_df["is_connected"]]
assert not_connected.empty, (
    f"not connected: {not_connected[['index', 'inst_name']].to_dict('records')}"
)

for idx, signal in all_signal.items():
    res_type = identification.resonators[idx].res_type
    if res_type == "shunt":
        assert signal.reaches_pad, (
            f"[{idx}] shunt must reach signal pad: {signal.drc_violations}"
        )
        assert signal.connector.shape_name in PAD_ROUTE_SHAPES, (
            f"[{idx}] unexpected shunt connector shape {signal.connector.shape_name!r}"
        )
        assert len(signal.connector.centerline) >= 2, (
            f"[{idx}] shunt connector must have a pad-directed route"
        )
    elif res_type == "series":
        assert signal.connector.shape_name == "on_resonator", (
            f"[{idx}] series must not pad-route (got {signal.connector.shape_name!r})"
        )
        assert len(signal.connector.centerline) >= 2, (
            f"[{idx}] series must have a perimeter arc centerline"
        )
        assert signal.n_net_polygons > 0, f"[{idx}] series perimeter strip is empty"
        strip_area = abs(signal.net_polygons[0].area())
        assert strip_area < 2000.0, (
            f"[{idx}] series strip too large ({strip_area:.0f} µm²)"
        )
    else:
        raise AssertionError(f"[{idx}] unsupported res_type {res_type!r}")

drc_flags = signal_overview_df[signal_overview_df["drc_violations"] > 0]

print(f"\nAll {len(all_signal)} resonators: connectivity checks passed")
print("  shunt: pad route + reaches_pad")
print("  series: on_resonator perimeter strip between release holes")
if not drc_flags.empty:
    print(f"DRC flags on {len(drc_flags)} resonator(s) (MTE vs ground MBE):")
    display(
        drc_flags[
            ["index", "inst_name", "res_type", "shape", "min_ground_spacing_um", "drc_violations"]
        ]
    )
else:
    print("No MTE/ground DRC flags")


In [ ]:
from src.export_gds import export_summary_df
from src.rteg_signal import export_signal_rteg_gds

MTE_OUT = DRAFT / "MTE_generated"

mte_export_df = export_summary_df(
    export_signal_rteg_gds(
        rteg_assemblies,
        all_signal,
        MTE_OUT,
        layermap=layermap,
        parent=parent,
    )
)

print(f"Exported {len(mte_export_df)} MTE GDS files to {MTE_OUT}")
print("Open each .gds with its matching .lyp in KLayout")
display(mte_export_df)

## 6. Verification - with clean drc layout

give it a real layout to verify if its accurate   